# Day C5 — 위상과 연결성: Hilbert, ITPC, PLV, 위상-진폭 결합

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C5.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — filter-Hilbert로 포락선과 위상 구하기

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(5)
fs = 500
t = np.arange(0, 3, 1 / fs)
true_amp = 1 + 0.8 * np.sin(2 * np.pi * 0.5 * t)          # 천천히 커졌다 작아지는 진폭
x = true_amp * np.cos(2 * np.pi * 10 * t) + rng.normal(0, 0.5, len(t))

# (1) 대역 통과(8–12 Hz) -> (2) Hilbert로 analytic signal -> 진폭과 위상
sos = signal.butter(4, [8, 12], btype="bandpass", fs=fs, output="sos")
xf = signal.sosfiltfilt(sos, x)
an = signal.hilbert(xf)
env = np.abs(an)                    # 순간 진폭(포락선)
ph = np.angle(an)                   # 순간 위상 (-π ~ π)

inner = slice(fs // 2, -fs // 2)    # 양 끝 0.5초는 가장자리 효과가 있어 제외
print("포락선과 참 진폭의 상관: %.3f" % np.corrcoef(env[inner], true_amp[inner])[0, 1])
print("analytic signal의 실수부 == 필터된 신호 ?", np.allclose(an.real, xf))
# 위상이 1초에 몇 바퀴 도는가 = 순간 주파수
inst_f = np.diff(np.unwrap(ph)) * fs / (2 * np.pi)
print("순간 주파수 중앙값: %.2f Hz" % np.median(inst_f[inner]))

### 예시 2 — 위상 리셋과 ITPC, 그리고 시행 수의 바닥값

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(6)
fs = 250
t = np.arange(-0.5, 1.0, 1 / fs)
sos = signal.butter(4, [4, 8], btype="bandpass", fs=fs, output="sos")

def make_trials(n_tr, reset=True):
    X = np.zeros((n_tr, len(t)))
    for k in range(n_tr):
        ph0 = rng.uniform(0, 2 * np.pi)
        pre = np.sin(2 * np.pi * 6 * t + ph0)                       # 자극 전: 시행마다 위상 제각각
        post = np.sin(2 * np.pi * 6 * t) if reset else pre          # 자극 후: 위상이 0으로 '리셋'되는가
        X[k] = np.where(t < 0, pre, post) + rng.normal(0, 1, len(t))
    return X

def itpc(X):
    ph = np.angle(signal.hilbert(signal.sosfiltfilt(sos, X, axis=1), axis=1))
    return np.abs(np.mean(np.exp(1j * ph), axis=0))                  # 시행에 걸친 평균 단위벡터의 길이

for reset in [True, False]:
    v = itpc(make_trials(50, reset))
    print("위상 리셋 %-5s: ITPC 자극 전(-0.4~-0.1초) %.2f | 자극 후(0.2~0.6초) %.2f" %
          (reset, v[(t > -0.4) & (t < -0.1)].mean(), v[(t > 0.2) & (t < 0.6)].mean()))

# 위상이 완전히 무작위여도 시행 수가 적으면 ITPC가 0보다 크게 나온다
for n in [10, 40, 160]:
    sims = [np.abs(np.mean(np.exp(1j * rng.uniform(0, 2 * np.pi, n)))) for _ in range(2000)]
    print("무작위 위상, 시행 %3d개: ITPC 평균 %.3f (참고: sqrt(pi/4/n) = %.3f)" % (n, np.mean(sims), np.sqrt(np.pi / 4 / n)))

### 예시 3 — PLV, PLI, coherence와 공통 원천

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(7)
fs = 250
n = fs * 20
t = np.arange(n) / fs
sos = signal.butter(4, [8, 12], btype="bandpass", fs=fs, output="sos")

def phase(x):
    return np.angle(signal.hilbert(signal.sosfiltfilt(sos, x)))

def narrow_osc():
    """주파수가 조금씩 흔들리는 10 Hz 리듬 (실제 뇌 리듬처럼)"""
    f_inst = 10 + np.convolve(rng.normal(0, 1, n), np.ones(fs) / fs, mode="same") * 20
    return np.sin(2 * np.pi * np.cumsum(f_inst) / fs)

def plv_and_wpli(a, b):
    d = phase(a) - phase(b)
    plv = np.abs(np.mean(np.exp(1j * d)))
    im = np.imag(np.exp(1j * d))        # 위상차의 사인 = 0이 아닌 시차가 있을 때만 0에서 벗어난다
    pli = np.abs(np.mean(np.sign(im)))  # phase lag index
    return plv, pli

src = narrow_osc()
cases = {
    "독립된 두 리듬": (narrow_osc() + rng.normal(0, 1, n), narrow_osc() + rng.normal(0, 1, n)),
    "한 리듬, 25 ms 시차": (src + rng.normal(0, 1, n), np.roll(src, int(0.025 * fs)) + rng.normal(0, 1, n)),
    "한 원천을 두 전극이 동시에 봄": (src + rng.normal(0, 1, n), 0.7 * src + rng.normal(0, 1, n)),
}
for name, (a, b) in cases.items():
    plv, pli = plv_and_wpli(a, b)
    f, coh = signal.coherence(a, b, fs, nperseg=2 * fs)
    print("%-18s PLV %.2f | PLI %.2f | 10 Hz coherence %.2f" % (name, plv, pli, coh[np.argmin(np.abs(f - 10))]))

### 예시 4 — theta 위상과 gamma 진폭의 결합

In [ ]:
import numpy as np
from scipy import signal

rng = np.random.default_rng(8)
fs = 1000
t = np.arange(0, 30, 1 / fs)
# 주파수가 대략 4–8 Hz 사이에서 천천히 흔들리는 theta (완벽한 사인파면 대리 검정이 작동하지 않는다)
f_inst = 6 + np.convolve(rng.normal(0, 1, len(t)), np.ones(fs) / fs, mode="same") * 15
theta = np.sin(2 * np.pi * np.cumsum(f_inst) / fs)

def make(coupled):
    gamma_amp = 1 + 0.8 * theta if coupled else np.ones_like(t)    # theta 마루에서 gamma가 커진다
    return theta + 0.3 * gamma_amp * np.sin(2 * np.pi * 60 * t) + rng.normal(0, 0.5, len(t))

def bp(x, lo, hi):
    return signal.sosfiltfilt(signal.butter(4, [lo, hi], btype="bandpass", fs=fs, output="sos"), x)

def mvl(x):
    ph = np.angle(signal.hilbert(bp(x, 4, 8)))       # 느린 리듬의 위상
    amp = np.abs(signal.hilbert(bp(x, 50, 70)))      # 빠른 리듬의 진폭
    return np.abs(np.mean(amp * np.exp(1j * ph))), ph, amp

for coupled in [True, False]:
    x = make(coupled)
    obs, ph, amp = mvl(x)
    # 대리(surrogate) 분포: 진폭 시계열을 무작위로 크게 밀어서(원형 이동) 위상과의 짝을 끊는다
    null = [np.abs(np.mean(np.roll(amp, rng.integers(fs, len(t) - fs)) * np.exp(1j * ph))) for _ in range(200)]
    z = (obs - np.mean(null)) / np.std(null)
    print("결합 %-5s: MVL %.4f | 대리 평균 %.4f | z = %.1f" % (coupled, obs, np.mean(null), z))